# ParaScape RNA-seq release snapshot and GitHub data audit, 2026-10-29


# RNA-seq release audit

All 296 completed study/run entries from 23 studies are represented. The full plan has 1,011 entries; pending runs are visible in the sample design but are not measurements. This executed notebook audits the transformed snapshot. Acquisition and quantification were run by the deterministic scripts; primary ENA XML retrievals and hashes are recorded in sample_metadata.json.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from parascape import paths, expression_contrast as X
from parascape.rna_profiles import snapshot
root = Path(paths.data_dir()) / "rnaseq"
info = snapshot(str(root))
assert info["verified_runs"] == 296
assert len(info["studies"]) == 23
{"verified_runs": info["verified_runs"], "studies": len(info["studies"]), "ME49_host_genes": info["genes"], "references": info["references"]}

{'verified_runs': 296, 'studies': 23, 'ME49_host_genes': 8164, 'references': ['ME49', 'ME49_host', 'GT1', 'VEG']}

## Verify every published output

Hashes retain source and transformed-output lineage. This is integrity verification, not biological validation or correction of cross-study batch effects.

In [2]:
for relative, expected in info["outputs"].items():
    assert hashlib.sha256((root / relative).read_bytes()).hexdigest() == expected
print(len(info["outputs"]), "snapshot outputs verified")
pd.DataFrame(info["studies"])

234 snapshot outputs verified


,study,completed_runs,planned_runs
0,GSE103113,6,6
1,GSE116539,4,4
2,GSE125120,12,12
3,GSE132238,16,16
4,GSE136123,18,18
5,GSE139537,15,15
6,GSE140197,9,9
7,GSE150334,6,6
8,GSE156685,8,8
9,GSE158231,14,14


## Submitted samples, controls and repeats

BioSample/run links do not establish independent biological replication. Source-declared biological replicate labels are retained; matching replicate numbers alone do not establish pairing.

In [3]:
design = pd.read_csv(root / "sample_design.tsv", sep="\t")
assert len(design) == 1011
assert (design.processing_status == "verified").sum() == 296
priority = design[design.study == "PRJNA1008424"]
assert len(priority) == 8
priority[["study", "sample_alias", "sample_title", "replicate_type", "control_role", "processing_status"]]

,study,sample_alias,sample_title,replicate_type,control_role,processing_status
695,PRJNA1008424,RHkoTrx1_Rep1,Trx1 mutant_1,biological (source attribute),not assigned,pending
696,PRJNA1008424,RHkoTrx1_Rep3,Trx1 mutant_3,biological (source attribute),not assigned,pending
697,PRJNA1008424,RH_Rep3,wild type control_3,biological (source attribute),control (source label),pending
698,PRJNA1008424,RH_Rep1,wild type control_1,biological (source attribute),control (source label),pending
699,PRJNA1008424,RHkoTrx1_Rep2,Trx1 mutant_2,biological (source attribute),not assigned,pending
700,PRJNA1008424,RHkoTrx1_Rep4,Trx1 mutant_4,biological (source attribute),not assigned,pending
701,PRJNA1008424,RH_Rep2,wild type control_2,biological (source attribute),control (source label),pending
702,PRJNA1008424,RH_Rep4,wild type control_4,biological (source attribute),control (source label),pending


## Reproduction

Run scripts/build_rna_snapshot.py with the explicit NAS root, frozen plan, verified.json and a new output directory. Then run scripts/extend_rna_references.py for all already-verified alternative conventions and scripts/enrich_rna_samples.py for source annotations. Finally scripts/build_data_release.py creates the deterministic GitHub asset and the package descriptor. The original full-precision quantifications remain in the processing archive; the release matrices use float32 and preserve missing values separately from zeros. No prediction features or calibrated models are changed by this release.